In [1]:
import pandas as pd
import plotly.express as px
from pathlib import Path

RAW = Path("../data/raw")
file = next(RAW.glob("*kldb2010-zeitreihe-d*.xlsx"))

raw = pd.read_excel(file, sheet_name="1", header=None, skiprows=9).iloc[:, :40]
names = pd.read_excel(file, sheet_name="1", header=None, nrows=8).iloc[7, 3:40].tolist()
names = [" ".join(str(n).replace("-", "").split()) for n in names]   # чистим переносы в названиях
raw.columns = ["year", "quarter", "total"] + names
raw["year"] = pd.to_numeric(raw["year"].astype(str).str.strip(), errors="coerce").ffill()  # год есть только в первой строке
raw["quarter"] = raw["quarter"].astype(str).str.strip()
emp = raw.dropna(subset=["total"])
emp.tail()

,year,quarter,total,"11 Land, Tier, Forstwirtschaftsberufe",12 Gartenbauberufe und Floristik,"21 Rohstoffgewinn und aufbereitung, Glas und Keramikherstellung und verarbeitung","22 Kunststoffherstellung und verarbeitung, Holzbe und verarbeitung","23 Papier und Druckberufe, technische Mediengestaltung","24 Metallerzeugung und bearbeitung, Metallbauberufe",25 Maschinen und Fahrzeugtechnikberufe,...,73 Berufe in Recht und Verwaltung,81 Medizinische Gesundheitsberufe,"82 Nicht medizinische Gesundheits, Körperpflege und Wellnessberufe, Medizintechnik","83 Erziehung, soziale und hauswirtschaftliche Berufe, Theologie",84 Lehrende und ausbildende Berufe,"91 Sprach, literatur, geistes, gesellschafts und wirtschaftswissenschaftliche Berufe","92 Werbung, Marketing, kaufmännische und redaktionelle Medienberufe","93 Produktdesign und kunsthandwerkliche Berufe, bildende Kunst, Musikinstrumentenbau","94 Darstellende, unterhaltende Berufe",Keine Zuordnung möglich
49,2025.0,März,34887706.0,234413.0,274480.0,108993.0,481066.0,244372.0,1099964.0,1822134.0,...,1261645.0,2926444.0,976427.0,2199381.0,754446.0,116759.0,657975.0,61417.0,132194.0,172258
50,2025.0,Juni,34885488.0,247256.0,279359.0,109334.0,477009.0,241478.0,1092927.0,1811903.0,...,1261878.0,2923892.0,978128.0,2197486.0,758681.0,116685.0,655169.0,60660.0,132050.0,170182
51,2025.0,September,35215934.0,254132.0,281934.0,109413.0,477709.0,240663.0,1092558.0,1833024.0,...,1283506.0,2978125.0,994705.0,2230220.0,751176.0,116251.0,654093.0,60939.0,134419.0,173777
52,2025.0,Dezember,34985555.0,225242.0,266410.0,105775.0,468806.0,237475.0,1069518.0,1809021.0,...,1285729.0,2999788.0,998188.0,2248783.0,758959.0,116854.0,650090.0,60265.0,129464.0,171988
53,2026.0,März,34825934.0,233979.0,273799.0,106167.0,465278.0,234156.0,1060892.0,1792926.0,...,1281719.0,2993881.0,994289.0,2249650.0,749229.0,116782.0,648401.0,59387.0,133047.0,167106


In [2]:
june = emp[emp["quarter"] == "Juni"].set_index("year").drop(columns="quarter").astype(float)
change = pd.DataFrame({"jobs_2015": june.loc[2015], "jobs_2025": june.loc[2025]})
change["change_abs"] = change["jobs_2025"] - change["jobs_2015"]
change["change_pct"] = ((change["jobs_2025"] / change["jobs_2015"] - 1) * 100).round(1)
avg = change.loc["total", "change_pct"]
groups = change.drop(index=["total", "Keine Zuordnung möglich"]).sort_values("change_pct")
groups

,jobs_2015,jobs_2025,change_abs,change_pct
28 Textil und Lederberufe,127502.0,103567.0,-23935.0,-18.8
"24 Metallerzeugung und bearbeitung, Metallbauberufe",1320438.0,1092927.0,-227511.0,-17.2
"21 Rohstoffgewinn und aufbereitung, Glas und Keramikherstellung und verarbeitung",127724.0,109334.0,-18390.0,-14.4
"23 Papier und Druckberufe, technische Mediengestaltung",277360.0,241478.0,-35882.0,-12.9
"22 Kunststoffherstellung und verarbeitung, Holzbe und verarbeitung",533794.0,477009.0,-56785.0,-10.6
"93 Produktdesign und kunsthandwerkliche Berufe, bildende Kunst, Musikinstrumentenbau",62932.0,60660.0,-2272.0,-3.6
62 Verkaufsberufe,2056653.0,2061079.0,4426.0,0.2
25 Maschinen und Fahrzeugtechnikberufe,1751856.0,1811903.0,60047.0,3.4
33 (Innen) Ausbauberufe,367239.0,385029.0,17790.0,4.8
"72 Berufe der Finanzdienstleistungen, Rechnungswesen und Steuerberatung",1385023.0,1463116.0,78093.0,5.6


In [3]:
groups["trend"] = groups["change_pct"].apply(
    lambda x: "shrinking" if x < 0 else ("below average" if x < avg else "above average"))
plot = groups.reset_index(names="occupation_group")

fig = px.bar(plot, x="change_pct", y="occupation_group", orientation="h", color="trend",
             color_discrete_map={"shrinking": "#d62728", "below average": "#bbbbbb", "above average": "#1f77b4"},
             text="change_pct", height=900,
             title=f"Employment change 2015–2025 by occupation group (Germany average: +{avg}%)",
             labels={"change_pct": "Change in employment, %", "occupation_group": ""})
fig.add_vline(x=avg, line_dash="dash")
fig.show()
fig.write_html("../output/charts/02_shrinking_growing.html")

## Finding 2: Shrinking and growing occupations (2015–2025)

- Total employment grew by **+13.4%**, but **6 occupation groups shrank**, all in material production: textiles (−18.8%), metalworking (−17.2%, −228,000 jobs), raw materials, paper/printing, plastics/wood.
- Sales occupations stagnated (+0.2%) — relatively shrinking.
- Fastest growth: **IT (+67.2%)**, humanities/social sciences (+54.5%), **education & social work (+43.8%, +669,000 jobs — the largest absolute gain)**, healthcare (+24.6%).
- Open question: IT employment grows strongly while the BA no longer sees an IT shortage. Is the market saturating?

Source: BA, Beschäftigte nach Berufen (KldB 2010), June 2015 vs June 2025.

In [5]:
fig.update_layout(template="plotly_white", width=1300, height=950)
fig.write_image("../output/charts/02_shrinking_growing.png", scale=2)
fig.write_html("../output/charts/02_shrinking_growing.html")